In [1]:
from pyspark.sql import SparkSession

In [2]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Read HDFS Example") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

df = spark.read.csv("/linkedin-data/linkedin_job_postings.csv", header=True)

df.show(5)


+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|            job_link| last_processed_time|got_summary|got_ner|is_being_worked|           job_title|             company|        job_location|first_seen|search_city|search_country|     search_position| job_level|job_type|
+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|https://www.linke...|2024-01-21 07:12:...|          t|      t|              f|Account Executive...|                  BD|       San Diego, CA|2024-01-15|   Coronado| United States|         Color Maker|Mid senior|  Onsite|
|https://www.linke...|2024-01-21 07:39:...|          t|      t|              f|Registered Nurse ...|   Trinity H

This code initializes Spark in local mode and points it to the HDFS cluster running inside Docker. Setting spark.hadoop.fs.defaultFS tells Spark: use HDFS as your main filesystem, and the URL hdfs://namenode:9000 is how Spark locates the NameNode service inside Docker. So when we later call spark.read, Spark will load data directly from HDFS.

In [3]:
df.printSchema()

root
 |-- job_link: string (nullable = true)
 |-- last_processed_time: string (nullable = true)
 |-- got_summary: string (nullable = true)
 |-- got_ner: string (nullable = true)
 |-- is_being_worked: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- company: string (nullable = true)
 |-- job_location: string (nullable = true)
 |-- first_seen: string (nullable = true)
 |-- search_city: string (nullable = true)
 |-- search_country: string (nullable = true)
 |-- search_position: string (nullable = true)
 |-- job_level: string (nullable = true)
 |-- job_type: string (nullable = true)



### Basic Data Processing with PySpark

#### 1. Select Columns

In [5]:
df.select("job_title", "company", "job_location").show(5)

+--------------------+--------------------+--------------------+
|           job_title|             company|        job_location|
+--------------------+--------------------+--------------------+
|Account Executive...|                  BD|       San Diego, CA|
|Registered Nurse ...|   Trinity Health MI|   Norton Shores, MI|
|RESTAURANT SUPERV...|Wasatch Adaptive ...|           Sandy, UT|
|Independent Real ...|Howard Hanna | Ra...|Englewood Cliffs, NJ|
|Group/Unit Superv...|IRS, Office of Ch...|        Chamblee, GA|
+--------------------+--------------------+--------------------+
only showing top 5 rows


#### 2. Filter Rows

In [10]:
df.filter(df.job_location == "Singapore").show(5)

+--------------------+--------------------+-----------+-------+---------------+---------+-----------------+------------+----------+------------+--------------+---------------+----------+--------+
|            job_link| last_processed_time|got_summary|got_ner|is_being_worked|job_title|          company|job_location|first_seen| search_city|search_country|search_position| job_level|job_type|
+--------------------+--------------------+-----------+-------+---------------+---------+-----------------+------------+----------+------------+--------------+---------------+----------+--------+
|https://sg.linked...|2024-01-19 15:50:...|          t|      t|              f| Operator|Unison Consulting|   Singapore|2024-01-17|Saint Joseph| United States|  Unit Operator|Mid senior|  Onsite|
+--------------------+--------------------+-----------+-------+---------------+---------+-----------------+------------+----------+------------+--------------+---------------+----------+--------+



In [8]:
df.filter(df.job_title.contains("Engineer")).show(5)

+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|            job_link| last_processed_time|got_summary|got_ner|is_being_worked|           job_title|             company|        job_location|first_seen|search_city|search_country|     search_position| job_level|job_type|
+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|https://ca.linked...|2024-01-21 08:08:...|          t|      t|              f|Engineering Proje...|Shared Health-Soi...|Winnipeg, Manitob...|2024-01-14|   Manitoba|        Canada| Program Coordinator|Mid senior|  Onsite|
|https://www.linke...|2024-01-21 08:08:...|          t|      t|              f|Control Systems I...|            

In [9]:
df.filter(df.job_title.ilike("%data%")).show(5)

+--------------------+--------------------+-----------+-------+---------------+--------------------+------------------+----------------+----------+-----------+--------------+--------------------+----------+--------+
|            job_link| last_processed_time|got_summary|got_ner|is_being_worked|           job_title|           company|    job_location|first_seen|search_city|search_country|     search_position| job_level|job_type|
+--------------------+--------------------+-----------+-------+---------------+--------------------+------------------+----------------+----------+-----------+--------------+--------------------+----------+--------+
|https://www.linke...|2024-01-19 09:45:...|          f|      f|              f|Senior Manager of...|      ClickJobs.io|Reynoldsburg, OH|2024-01-16|   Columbus| United States|        Manager Camp|Mid senior|  Onsite|
|https://www.linke...|2024-01-19 09:45:...|          f|      f|              f|Database Manageme...|     ClearanceJobs|  Washington, DC|

#### 3. Create New Columns

In [12]:
from pyspark.sql.functions import lower, trim

df2 = df.withColumn("job_title_clean", lower(trim(df.job_title)))
df2.show(5)

+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+--------------------+
|            job_link| last_processed_time|got_summary|got_ner|is_being_worked|           job_title|             company|        job_location|first_seen|search_city|search_country|     search_position| job_level|job_type|     job_title_clean|
+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+--------------------+
|https://www.linke...|2024-01-21 07:12:...|          t|      t|              f|Account Executive...|                  BD|       San Diego, CA|2024-01-15|   Coronado| United States|         Color Maker|Mid senior|  Onsite|account executive...|
|https://www.linke...|2024-0

#### 4. Remove Columns

In [13]:
df2 = df2.drop("job_title_clean")
df2.show(5)

+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|            job_link| last_processed_time|got_summary|got_ner|is_being_worked|           job_title|             company|        job_location|first_seen|search_city|search_country|     search_position| job_level|job_type|
+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|https://www.linke...|2024-01-21 07:12:...|          t|      t|              f|Account Executive...|                  BD|       San Diego, CA|2024-01-15|   Coronado| United States|         Color Maker|Mid senior|  Onsite|
|https://www.linke...|2024-01-21 07:39:...|          t|      t|              f|Registered Nurse ...|   Trinity H

#### 5. GroupBy and Count

In [15]:
df.groupBy("job_location").count().show(20)

+--------------------+-----+
|        job_location|count|
+--------------------+-----+
|     Gainesville, FL| 1404|
|         Atwater, CA|   47|
|      Burlington, WA|   72|
|Vancouver, Britis...| 2804|
|       Jonesboro, AR|  334|
|       St George, UT|  210|
|     Maple Grove, MN|  291|
|     Leonardtown, MD|   76|
|          Dundee, MI|   26|
|    Barefoot Bay, FL|    2|
|         Malabar, FL|   20|
|            Novi, MI|  493|
|    College Park, GA|   74|
|  Greater Perth Area|  162|
|Sleaford, England...|   58|
|   St Petersburg, FL| 1335|
|          Mojave, CA|   84|
|      Warrington, PA|   66|
|      Pleasanton, CA|  577|
|Six Hills, Englan...|    2|
+--------------------+-----+
only showing top 20 rows


#### 6. Count and Group By Company and Job Type

In [16]:
df.groupBy("company", "job_type").count().show(30)

+--------------------+--------+-----+
|             company|job_type|count|
+--------------------+--------+-----+
|          McDonald's|  Onsite| 8125|
|   Places for People|  Onsite|   96|
|Pyramid Healthcar...|  Onsite|  156|
|Gateway Regional ...|  Onsite|   28|
|Supplemental Heal...|  Onsite|  357|
|    Nisga'a Tek, LLC|  Onsite|   13|
|       Hyatt Centric|  Onsite|   30|
|Loblaw Companies ...|  Onsite|  346|
|Oldcastle Buildin...|  Onsite|   26|
|        Allianz Life|  Onsite|   17|
|Aramark Refreshments|  Onsite|  116|
|            Team JDC|  Onsite|    2|
|          Sam's Club|  Onsite|  326|
|Aspire Indiana He...|  Onsite|   30|
|     TPF Recruitment|  Onsite|  102|
|           TECHOHANA|  Onsite|   10|
|   American Traveler|  Onsite|  170|
|      UK Home Office|  Onsite|    2|
|Goodwill of Weste...|  Onsite|   43|
|            West Elm|  Onsite|   35|
|           MTM, Inc.|  Onsite|  140|
|    Artius Solutions|  Onsite|  437|
|SkyePoint Decisio...|  Onsite|    7|
|        CoS

In [17]:
df.orderBy(df.last_processed_time.desc()).show(10)

+--------------------+--------------------+--------------------+----------+---------------+--------------+--------------------+------------+----------+-----------+--------------+---------------+---------+--------+
|            job_link| last_processed_time|         got_summary|   got_ner|is_being_worked|     job_title|             company|job_location|first_seen|search_city|search_country|search_position|job_level|job_type|
+--------------------+--------------------+--------------------+----------+---------------+--------------+--------------------+------------+----------+-----------+--------------+---------------+---------+--------+
|2 Senior Forestry...|  State of Wisconsin|         Madison, WI|2024-01-14|        Windsor| United States|Educational Resou...|  Mid senior|    Onsite|       NULL|          NULL|           NULL|     NULL|    NULL|
|                 ...|RED Engineering D...|Newcastle upon Ty...|2024-01-12|       Tyneside|United Kingdom|             Charter|  Mid senior|    

#### 7. Handle Null Values

Add "unknown" text to NA 

In [22]:
df.na.fill("Unknown").show(5)

+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|            job_link| last_processed_time|got_summary|got_ner|is_being_worked|           job_title|             company|        job_location|first_seen|search_city|search_country|     search_position| job_level|job_type|
+--------------------+--------------------+-----------+-------+---------------+--------------------+--------------------+--------------------+----------+-----------+--------------+--------------------+----------+--------+
|https://www.linke...|2024-01-21 07:12:...|          t|      t|              f|Account Executive...|                  BD|       San Diego, CA|2024-01-15|   Coronado| United States|         Color Maker|Mid senior|  Onsite|
|https://www.linke...|2024-01-21 07:39:...|          t|      t|              f|Registered Nurse ...|   Trinity H

In [23]:
# drop NA value
# df.na.drop()